# Topic Merging via Similarity Analysis
<!-- structured-notebook -->
## Notebook Summary
Purpose: after BERTopic assignment, identify semantically similar topic pairs and merge them to produce a cleaner, more interpretable topic structure.

Main steps:
- Build a sparse document-topic soft-membership matrix via `topic_model.transform()` + top-k retention.
- Compute three complementary similarity signals (co-activation, centroid cosine, Jensen-Shannon divergence).
- Combine into a composite score; form connected components over high-scoring pairs to produce merge groups.


# Inputs / Outputs
<!-- io-table -->

| Role | File | Upstream / downstream |
|---|---|---|
| Input | `<DATA_ROOT>/Reddit/output/bertopic/round_{ROUND}/bertopic_no_embed/` | Produced by preprocessing |
| Input | `<DATA_ROOT>/Reddit/output/bertopic/round_{ROUND}/embeddings_fp32_l2.npy` | Produced by preprocessing |
| Output | `<DATA_ROOT>/Reddit/output/bertopic/round_{ROUND}/merged_topics.csv` | Merged topic proposals; consumed by BTM |


# Three Similarity Signals
<!-- structured-notebook -->
## Summary

1. **Co-activation matrix.** Measures how often two topics share documents via soft membership. Let $X$ be the sparse document-topic membership matrix of shape $(M, K)$ with rows normalized to sum to 1 over retained topics. Then
$$
C = X^\top X, \quad \text{cos\_doc}_{ij} = \frac{C_{ij}}{\sqrt{C_{ii}} \cdot \sqrt{C_{jj}}}
$$

2. **Centroid cosine similarity.** Computes weighted centroids of topic clusters in embedding space and measures angular proximity between centroids. Captures semantic similarity in the original embedding space.

3. **Jensen-Shannon divergence (JSD).** Symmetric, bounded measure of distributional similarity between the keyword distributions of two topics. Lower JSD indicates more similar word usage.

### Hyperparameters

- `TOP_K` — number of topic assignments retained per document (typical: 2)
- `PROB_MIN` — minimum probability for retained assignments (typical: 0.15)
- The top-1 assignment is always kept regardless of `PROB_MIN`.

Tune these two when the soft-membership matrix looks too sparse (raise `TOP_K`) or too noisy (raise `PROB_MIN`).


## Setup & Imports

In [ ]:
import sys
from pathlib import Path
# Walk up from the notebook's directory to the repo root (the first ancestor that contains `src/`) and add it to sys.path
_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

In [ ]:
import json
import pickle
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.spatial.distance import jensenshannon
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
from bertopic import BERTopic

## Configuration

Set paths to the saved BERTopic model, preprocessed documents, and precomputed embeddings. Adjust `TOP_K` and `PROB_MIN` to control how many topics each document can belong to in the soft membership matrix.

In [ ]:
# --- Paths ---
MODEL_PATH = "../main_topic_models/bertopic_all-mpnet-base-v2_final_output/bertopic_final.model"
DOCS_PKL   = "../preprocessed_docs/preprocessed_reddit_docs.pkl"
EMB_NPY    = "../preprocessed_docs/reddit_embeddings.npy"  # embeddings aligned 1:1 with DOCS_PKL

# --- Parameters ---
BATCH    = 50_000   # batch size for streaming transform
TOP_K    = 2        # number of top topic slots per document
PROB_MIN = 0.15     # minimum probability to retain a topic assignment

# --- Output ---
OUTPUT_DIR = "../scripts/topic_cleaning_and_analysis/topic_narrowing-saved_files"

## Load Model and Documents

Load the trained BERTopic model and the preprocessed document corpus. The embeddings file is memory-mapped so it does not need to fit entirely in RAM.

In [ ]:
topic_model = BERTopic.load(MODEL_PATH)

with open(DOCS_PKL, "rb") as f:
    all_docs = pickle.load(f)

print(f"Loaded {len(all_docs):,} documents")
print(f"Model has {len(topic_model.get_topics()) - 1} topics (excluding outlier topic -1)")

## Build Sparse Document-Topic Membership Matrix

For each document we run `topic_model.transform()` using precomputed embeddings and retain the top-$k$ topic assignments whose probability exceeds `PROB_MIN`. The top-1 assignment is always kept regardless of its probability.

The result is a sparse matrix $X$ of shape $(M, K)$ where $M$ is the number of documents and $K$ is the number of topics. Each row is normalised to sum to 1 over the retained topics, giving a soft membership distribution.

In [ ]:
def build_sparse_membership_from_embeddings(topic_model, all_docs, emb_npy_path,
                                            batch_size=50_000, top_k=3, prob_min=0.10):
    """Build a sparse (M x K) soft membership matrix from precomputed embeddings."""
    E = np.load(emb_npy_path, mmap_mode="r")
    M = len(all_docs)
    assert M == E.shape[0], f"Doc count ({M}) != embedding rows ({E.shape[0]})"

    rows_acc, cols_acc, vals_acc = [], [], []
    K = None

    for start in tqdm(range(0, M, batch_size), desc="Building membership matrix"):
        end = min(M, start + batch_size)
        docs = all_docs[start:end]
        embs = E[start:end]
        topics, probs = topic_model.transform(docs, embeddings=embs)
        if K is None:
            K = probs.shape[1]

        n, k = probs.shape

        # Always keep top-1
        top1_idx = probs.argmax(axis=1)
        top1_val = probs[np.arange(n), top1_idx]

        # Candidates for additional slots
        idx = np.argpartition(probs, -top_k, axis=1)[:, -top_k:]  # (n, top_k)
        vals = np.take_along_axis(probs, idx, axis=1)

        keep = vals >= prob_min
        # Ensure top-1 is included even if below threshold
        for r in range(n):
            for c in range(idx.shape[1]):
                if idx[r, c] == top1_idx[r]:
                    keep[r, c] = True

        r = np.repeat(np.arange(n), idx.shape[1]).reshape(n, -1)[keep].reshape(-1)
        c = idx[keep].reshape(-1)
        v = vals[keep].reshape(-1).astype(np.float32)

        # Normalize per-row over kept entries
        row_sums = np.zeros(n, dtype=np.float32)
        np.add.at(row_sums, r, v)
        v = v / np.maximum(row_sums[r], 1e-9)

        rows_acc.append(r + start)
        cols_acc.append(c)
        vals_acc.append(v)

    rows = np.concatenate(rows_acc) if rows_acc else np.array([], dtype=np.int64)
    cols = np.concatenate(cols_acc) if cols_acc else np.array([], dtype=np.int64)
    vals = np.concatenate(vals_acc) if vals_acc else np.array([], dtype=np.float32)

    X = coo_matrix((vals, (rows, cols)), shape=(M, K), dtype=np.float32).tocsr()
    return X

In [ ]:
X = build_sparse_membership_from_embeddings(
    topic_model,
    all_docs,
    emb_npy_path=EMB_NPY,
    batch_size=BATCH,
    top_k=TOP_K,
    prob_min=PROB_MIN,
)

print(f"Membership matrix shape: {X.shape}")
print(f"Non-zero entries: {X.nnz:,}")
print(f"Rows with >= 1 topic: {(np.diff(X.indptr) > 0).sum():,} of {X.shape[0]:,}")

col_mass = np.asarray(X.sum(axis=0)).ravel()
print(f"Topics with zero mass: {(col_mass == 0).sum()}")

## Build Co-activation Matrix

The co-activation matrix $C = X^\top X$ captures how much membership mass two topics share across documents. Entry $C_{ij}$ is high when many documents are assigned to both topic $i$ and topic $j$.

We normalise $C$ into a cosine similarity matrix:

$$
\text{cos\_doc}_{ij} = \frac{C_{ij}}{\sqrt{C_{ii}} \cdot \sqrt{C_{jj}}}
$$

This gives a value in $[0, 1]$ measuring the document-overlap similarity between each pair of topics.

In [ ]:
def coactivation_and_doc_cosine(X):
    """Compute the co-activation matrix and its cosine-normalised form."""
    C = (X.T @ X).astype(np.float32)  # K x K
    diag = np.sqrt(C.diagonal() + 1e-9)
    cos = C.copy()
    cos = cos.multiply(1.0 / diag).T.multiply(1.0 / diag).T
    return C, cos

In [ ]:
C, cos_doc = coactivation_and_doc_cosine(X)
K = cos_doc.shape[0]
print(f"Number of topics (K): {K}")

# Quick stats on off-diagonal cosine similarities
cd = cos_doc if isinstance(cos_doc, np.ndarray) else cos_doc.toarray()
tri = np.triu_indices(cd.shape[0], 1)
print(f"doc_cos off-diagonal: min={float(cd[tri].min()):.4f}, max={float(cd[tri].max()):.4f}")
print(f"Pairs with doc_cos >= 0.10: {int((cd[tri] >= 0.10).sum())}")

## Map Probability Column Index to Topic ID

The probability matrix returned by `topic_model.transform()` has columns indexed 0..K-1, but the actual BERTopic topic IDs may differ (e.g., non-contiguous after reductions). We infer the mapping by running a sample of documents through `transform()` and using argmax alignment: for each column index, the most frequently assigned topic ID wins.

In [ ]:
def infer_index_to_topic_id_via_argmax(topic_model, all_docs, emb_npy_path,
                                       batch_size=50_000, max_docs=300_000, skip_outlier=True):
    """
    Infer probability column index -> topic_id without get_document_info.
    Streams up to `max_docs`, passes precomputed embeddings, and uses argmax alignment.
    """
    E = np.load(emb_npy_path, mmap_mode="r")
    N = min(len(all_docs), E.shape[0])
    assert N == len(all_docs), "Embeddings and docs must align 1:1."

    counts = defaultdict(Counter)  # idx -> Counter({topic_id: votes})
    K = None
    seen = set()

    for start in tqdm(range(0, min(N, max_docs), batch_size), desc="Mapping index -> topic_id"):
        end = min(N, start + batch_size)
        docs = all_docs[start:end]
        embs = E[start:end]

        topics, probs = topic_model.transform(docs, embeddings=embs)
        if K is None:
            K = probs.shape[1]

        idx_max = probs.argmax(axis=1)
        for idx, tid in zip(idx_max, topics):
            tid = int(tid)
            if skip_outlier and tid == -1:
                continue
            counts[int(idx)][tid] += 1
            if sum(counts[int(idx)].values()) >= 3:
                seen.add(int(idx))

        # Stop early when most columns have sufficient votes
        if K is not None and len(seen) >= K - (1 if skip_outlier else 0):
            break

    # Pick the modal topic_id per index
    index_to_tid = {}
    assigned = set()
    for idx in range(K):
        c = counts.get(idx, None)
        if c and len(c) > 0:
            tid = c.most_common(1)[0][0]
            index_to_tid[idx] = tid
            assigned.add(tid)

    # Fill unmapped indices with unused topic_ids (rare edge case)
    all_topic_ids = [t for t in topic_model.get_topics().keys() if t not in (-1, None)]
    unused_tids = [t for t in all_topic_ids if t not in assigned]
    missing = [i for i in range(K) if i not in index_to_tid]
    for i, t in zip(missing, unused_tids):
        index_to_tid[i] = int(t)

    if len(index_to_tid) != K:
        print(f"Warning: mapped {len(index_to_tid)}/{K} indices. "
              f"Increase max_docs or lower batch_size to gather more votes.")

    return index_to_tid

In [ ]:
index_to_tid = infer_index_to_topic_id_via_argmax(
    topic_model,
    all_docs,
    EMB_NPY,
    batch_size=50_000,
    max_docs=300_000,
    skip_outlier=True,
)

print(f"Mapped {len(index_to_tid)} column indices to topic IDs")
print("Sample mapping (index -> topic_id):")
for idx in sorted(index_to_tid.keys())[:10]:
    print(f"  column {idx} -> topic {index_to_tid[idx]}")

## Compute Centroid Cosine Similarity

For each topic, we compute a weighted centroid in the original embedding space. The weight for document $d$ in topic $k$ is $X_{dk}$ (from the membership matrix). We then measure cosine similarity between all pairs of centroids.

This captures whether two topics occupy similar regions of the semantic embedding space, independent of the word-level features.

In [ ]:
def centroid_cosine_from_embeddings(X, emb_npy, chunk=100_000):
    """Stream embeddings (memmap) and accumulate weighted centroids per topic."""
    E = np.load(emb_npy, mmap_mode="r")
    M, D = E.shape
    K = X.shape[1]
    sums = np.zeros((K, D), dtype=np.float32)
    mass = np.zeros(K, dtype=np.float32)

    X_csr = X.tocsr()
    start = 0
    while start < M:
        end = min(M, start + chunk)
        Es = E[start:end]
        Xs = X_csr[start:end].tocoo()
        for r, c, v in zip(Xs.row, Xs.col, Xs.data):
            sums[c] += v * Es[r]
            mass[c] += v
        start = end

    mass = np.maximum(mass, 1e-9)
    centroids = sums / mass[:, None]

    # Cosine similarity between centroids
    norms = np.linalg.norm(centroids, axis=1, keepdims=True) + 1e-9
    centroids_normed = centroids / norms
    return (centroids_normed @ centroids_normed.T).astype(np.float32)

In [ ]:
centroid_cos = centroid_cosine_from_embeddings(X, EMB_NPY, chunk=100_000)

cc = centroid_cos if isinstance(centroid_cos, np.ndarray) else centroid_cos.toarray()
print(f"Centroid cosine matrix shape: {cc.shape}")
print(f"Centroid cosine off-diagonal max: {float(cc[tri].max()):.4f}")

## Jensen-Shannon Divergence

The Jensen-Shannon divergence (JSD) is a symmetric, bounded measure derived from KL divergence. For two keyword distributions $P_i$ and $P_j$ over the vocabulary of the top-$n$ words:

$$
\text{JSD}(P_i \| P_j) = \frac{1}{2} D_{KL}(P_i \| M) + \frac{1}{2} D_{KL}(P_j \| M), \quad M = \frac{P_i + P_j}{2}
$$

Low JSD means the two topics use similar keywords with similar weights. We compute JSD for every topic pair and later convert it to a similarity score via $1 - \text{JSD}$.

In [ ]:
def keyword_jsd_matrix(topic_model, K, top_n=30, index_to_topic_id=None):
    """Build a K x K Jensen-Shannon divergence matrix from topic keyword distributions."""
    topic_ids = [t for t in topic_model.get_topics().keys() if t != -1]
    wid = {}  # word -> column index
    rows, cols, vals = [], [], []

    id_to_row = {tid: i for i, tid in enumerate(sorted(topic_ids))}

    for tid in topic_ids:
        words = topic_model.get_topic(tid)
        if not words:
            continue
        words = words[:top_n]
        s = sum(wt for _, wt in words) + 1e-9
        for w, wt in words:
            j = wid.setdefault(w, len(wid))
            rows.append(id_to_row[tid])
            cols.append(j)
            vals.append(wt / s)

    if not rows:
        return None

    M_id = coo_matrix(
        (vals, (rows, cols)),
        shape=(len(id_to_row), len(wid)),
        dtype=np.float32,
    ).tocsr()

    # Reorder rows to match probability column index order
    if index_to_topic_id is None:
        M_idx = M_id
    else:
        order = [id_to_row[index_to_topic_id[i]] for i in range(K)]
        M_idx = M_id[order]

    jsd = np.zeros((K, K), dtype=np.float32)
    for i in range(K):
        pi = M_idx.getrow(i).toarray().ravel()
        s = pi.sum()
        if s == 0:
            continue
        pi = pi / s
        for j in range(i + 1, K):
            pj = M_idx.getrow(j).toarray().ravel()
            t = pj.sum()
            if t == 0:
                continue
            pj = pj / t
            jsd_ij = jensenshannon(pi + 1e-12, pj + 1e-12, base=np.e) ** 2
            jsd[i, j] = jsd[j, i] = jsd_ij

    return jsd

In [ ]:
jsd = keyword_jsd_matrix(topic_model, K, top_n=30, index_to_topic_id=index_to_tid)

if jsd is not None:
    jj = jsd if isinstance(jsd, np.ndarray) else jsd.toarray()
    print(f"JSD matrix shape: {jj.shape}")
    print(f"1 - JSD off-diagonal max: {float((1 - jj)[tri].max()):.4f}")
else:
    print("No keyword distributions available for JSD computation.")

## Identify Merge Candidates

We fuse the three signals into a composite score for each topic pair $(i, j)$:

$$
\text{score}_{ij} = 0.45 \cdot \text{doc\_cos}_{ij} + 0.25 \cdot \text{centroid\_cos}_{ij} + 0.20 \cdot (1 - \text{JSD}_{ij})
$$

A pair passes the gate if it exceeds minimum thresholds on each individual signal **and** the composite score exceeds `score_min`. The thresholds below have been relaxed from defaults to capture more candidates for inspection.

In [ ]:
def propose_merges(cos_doc, centroid_cos=None, jsd=None,
                   doc_cos_min=0.30, centroid_min=0.70, jsd_max=0.35, score_min=0.70):
    """Score all topic pairs and return those passing the thresholds."""
    # Ensure dense arrays for indexing
    if hasattr(cos_doc, "toarray"):
        cos_doc = cos_doc.toarray().astype(np.float32)
    if centroid_cos is not None and hasattr(centroid_cos, "toarray"):
        centroid_cos = centroid_cos.toarray().astype(np.float32)
    if jsd is not None and hasattr(jsd, "toarray"):
        jsd = jsd.toarray().astype(np.float32)

    K = cos_doc.shape[0]
    edges = []
    for i in range(K):
        for j in range(i + 1, K):
            dcos = float(cos_doc[i, j])
            if dcos < doc_cos_min:
                continue
            ccos = float(centroid_cos[i, j]) if centroid_cos is not None else dcos
            if ccos < centroid_min:
                continue
            kval = 1.0 - (float(jsd[i, j]) if jsd is not None else 0.3)
            if jsd is not None and (1.0 - jsd[i, j]) < (1.0 - jsd_max):
                continue
            score = 0.45 * dcos + 0.25 * ccos + 0.20 * kval
            if score >= score_min:
                edges.append((i, j, score, dcos, ccos, kval))

    edges.sort(key=lambda x: x[2], reverse=True)
    return edges

In [ ]:
edges = propose_merges(
    cos_doc,
    centroid_cos=centroid_cos,
    jsd=jsd,
    doc_cos_min=0.10,
    centroid_min=0.60,
    jsd_max=0.45,
    score_min=0.60,
)

print(f"{len(edges)} candidate pairs after gating.")
print()
print("Top candidate merges (index_i, index_j, score, doc_cos, centroid_cos, 1-JSD):")
print("-" * 85)
for e in edges[:30]:
    i, j, score, dc, cc_val, ks = e
    print(
        f"  idx {i} (topic {index_to_tid[i]:>3d})  <->  "
        f"idx {j} (topic {index_to_tid[j]:>3d})  "
        f"score={score:.3f}  doc={dc:.3f}  cent={cc_val:.3f}  ksim={ks:.3f}"
    )

## Form Merge Groups via Connected Components

Pairs with a composite score above `score_cut` are connected into groups using a union-find structure. Each connected component represents a set of topics that should be merged into one.

In [ ]:
def connected_components_from_edges(edges, K, score_cut=0.70):
    """Union-find to cluster topic indices into merge groups."""
    parent = list(range(K))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]  # path compression
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for i, j, score, *_ in edges:
        if score >= score_cut:
            union(i, j)

    groups = defaultdict(list)
    for t in range(K):
        groups[find(t)].append(t)

    return [g for g in groups.values() if len(g) > 1]

In [ ]:
groups = connected_components_from_edges(edges, K, score_cut=0.70)

print(f"Found {len(groups)} merge group(s):")
print()
for i, g in enumerate(groups):
    topic_ids = [index_to_tid[x] for x in g]
    print(f"  Group {i + 1}: indices {g} -> topic IDs {topic_ids}")

## Save Merge Groups

Persist the identified merge groups to JSON for downstream use (e.g., applying the merges to the BERTopic model or re-labelling documents).

In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

output_path = os.path.join(OUTPUT_DIR, "merging_groups.json")
with open(output_path, "w") as f:
    json.dump([[int(x) for x in g] for g in groups], f, indent=2)

print(f"Merge groups saved to: {output_path}")

## Diagnostics Summary

Final summary statistics to verify the quality of the membership matrix and similarity computations.

In [ ]:
rows_nonzero = np.diff(X.indptr) > 0
print(f"Rows with >= 1 topic: {rows_nonzero.sum():,} of {X.shape[0]:,}")

col_mass = np.asarray(X.sum(axis=0)).ravel()
print(f"Topics with zero mass: {(col_mass == 0).sum()}")

print(f"\ndoc_cos: min/max (off-diag): {float(cd[tri].min()):.4f} / {float(cd[tri].max()):.4f}")
print(f"Pairs with doc_cos >= 0.10: {int((cd[tri] >= 0.10).sum())}")

print(f"centroid_cos off-diag max: {float(cc[tri].max()):.4f}")

if jsd is not None:
    print(f"1 - JSD off-diag max: {float((1 - jj)[tri].max()):.4f}")

print(f"\nTotal candidate edges: {len(edges)}")
print(f"Merge groups formed: {len(groups)}")

---
<!-- nav-footer -->

← [01 subtopic analysis](../../../../../SocialMedia/Reddit/notebooks/BERTopic/03_topic_refinement/01_subtopic_analysis.ipynb) | [Project Overview](../../../../../PROJECT_OVERVIEW.ipynb) | [03 topic ranking and narrowing](../../../../../SocialMedia/Reddit/notebooks/BERTopic/03_topic_refinement/03_topic_ranking_and_narrowing.ipynb) →
